# Atlas Trade — Unsloth Qwen3.5-27B Setup

**Runtime:** Google Colab Pro — A100 80GB  
**Model:** `unsloth/Qwen3.5-27B` (BF16, no quantization)  
**Context:** 16384 tokens

Run cells top to bottom on a fresh A100 runtime. On subsequent sessions, re-run from **Step 4** onwards (model is cached in HF cache after the first download).

## Configuration

Set your GitHub repo URL and Hugging Face token here, or export them as environment variables before running:

```python
import os
os.environ["PROJECT_REPO_URL"] = "https://github.com/<your-username>/unsloth-qwen35-trading-assistant.git"
os.environ["HF_TOKEN"] = "hf_..."
```

In [ ]:
import os

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Model seçimi ────────────────────────────────────────────────────────────
# Qwen3-32B: Qwen3.5-27B'ye göre daha güçlü reasoning, native thinking modu,
#            A100 80GB'de 4-bit ile ~18GB VRAM kullanır.
MODEL_NAME     = "unsloth/Qwen3-32B"
MAX_SEQ_LENGTH = 16384   # 16k context — reasoning zinciri için gerekli
DTYPE          = None    # auto-detect (BF16 on A100)
LOAD_IN_4BIT   = True    # ~18GB VRAM

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"

print(f"Model   : {MODEL_NAME}")
print(f"Context : {MAX_SEQ_LENGTH} tokens")
print(f"4-bit   : {LOAD_IN_4BIT}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

## Step 1 — Clone or update the repository

In [2]:
import subprocess, pathlib, sys

def _run(cmd: list[str], **kwargs) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    # Add repo to Python path so `src` is importable
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

Cloning into '/content/unsloth-qwen35-trading-assistant'...
Repo ready at /content/unsloth-qwen35-trading-assistant


## Step 2 — Install Unsloth

Uses the official runtime-aware auto-installer — automatically matches the active CUDA/Torch stack on this Colab runtime.

In [3]:
import subprocess, sys

def _pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args],
        capture_output=True, text=True
    )
    # Print last 2000 chars of output so failures are visible
    out = (result.stdout + result.stderr)[-2000:]
    print(out)
    return result.returncode

print("=== Installing Unsloth ===\n")
code = _pip("install", "unsloth")

if code != 0:
    print("\nStandard install failed, trying git source...")
    code = _pip(
        "install",
        "unsloth @ git+https://github.com/unslothai/unsloth.git",
        "--no-build-isolation",
    )

if code != 0:
    raise RuntimeError("Unsloth installation failed. See output above.")

print("\n✅ Unsloth installed successfully.")
print("\n⚠️  RESTART THE RUNTIME NOW, then re-run from Step 1 (skip this cell).")
print("   Runtime menu → Restart session")

=== Installing Unsloth ===

━━━━━━━━ 3.3/3.3 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 28.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing

## Step 3 — Hugging Face login

Required only if you plan to push LoRA adapters or access gated models. Skip if `HF_TOKEN` is empty.

In [ ]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set — skipping login (public model download will still work).")

No HF_TOKEN set — skipping login (public model download will still work).


## Step 4 — Load the model

First run downloads ~54 GB of BF16 weights. Subsequent runs load from the HF cache (~1-2 minutes).

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Model loaded: unsloth/Qwen3.5-27B
Max sequence length: 8192


## Step 5 — Chat function

`chat()` keeps a conversation history so Atlas Trade remembers the context within a session.  
Call `chat(reset=True)` to start a fresh conversation.

In [ ]:
import sys, re
sys.path.insert(0, REPO_DIR)

from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages

_conversation_history = []
_text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def _strip_thinking(text: str) -> str:
    """
    Qwen3 <think>...</think> bloğunu temizle, sadece asıl cevabı döndür.
    Thinking içeriği tamamen kaldırılır — kullanıcıya sade, temiz çıktı gider.
    """
    # <think>...</think> bloğunu (multiline) kaldır
    cleaned = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.DOTALL)
    return cleaned.strip()


def chat(
    user_message: str,
    market_context: str = "",
    reset: bool = False,
    thinking: bool = True,         # ← varsayılan AÇIK: model önce düşünür, sonra yazar
    max_new_tokens: int = 2048,    # ← düşünce zinciri için artırıldı
    temperature: float = 0.6,
    top_p: float = 0.95,
):
    """
    Atlas Trade ile sohbet et.

    Parametreler
    ------------
    user_message    : Sorun veya analiz isteğin.
    market_context  : fetch_context() veya fetch_multi_tf_context() çıktısı.
    reset           : True → geçmiş sıfırlanır, yeni konuya başlanır.
    thinking        : True → model önce sessiz muhakeme yapar (önerilen).
    max_new_tokens  : Maksimum üretilecek token (thinking dahil).
    """
    global _conversation_history
    if reset:
        _conversation_history = []
        print("[sıfırlandı]")

    messages = build_trading_messages(
        user_message=user_message,
        market_context=market_context,
        history=_conversation_history,
    )

    # Chat template — enable_thinking Qwen3 serisinde desteklenir
    try:
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking,
        )
    except TypeError:
        # Eski transformers sürümü enable_thinking bilmiyorsa sessizce geç
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = _text_tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids      = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        pad_token_id   = _text_tokenizer.eos_token_id,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = top_p,
        do_sample      = True,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw        = _text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response   = _strip_thinking(raw)

    _conversation_history.append({"role": "user",      "content": user_message})
    _conversation_history.append({"role": "assistant", "content": response})
    return response


def clear_history():
    global _conversation_history
    _conversation_history = []
    print("[geçmiş temizlendi]")


print("✅ chat() hazır")
print("   thinking modu: AÇIK (varsayılan)")
print("   max_new_tokens: 2048")

## Step 6 — Start trading

Atlas Trade hazır. Aşağıdaki hücreyi düzenleyerek analizini başlat.

## Step 7 — Live market data (Bybit)

`fetch_context()` Bybit'ten canlı OHLCV çeker, RSI / EMA / MACD hesaplar ve `chat()` için hazır bir `market_context` stringi döner.  
API key gerekmez — public endpoint.

In [ ]:
import importlib
import src.market_data as _md
importlib.reload(_md)
from src.market_data import fetch_context, fetch_multi_tf_context


def live_chat(
    user_message: str,
    symbol: str = "BTCUSDT",
    timeframe: str = "4h",
    multi_tf: bool = False,
    timeframes: list = None,
    reset: bool = False,
    **kwargs,
):
    """
    Bybit'ten canlı piyasa verisi çekip otomatik olarak chat()'e gönderir.

    Parametreler
    ------------
    user_message : Analiz sorun.
    symbol       : Bybit sembolü, örn. "BTCUSDT", "ETHUSDT", "SOLUSDT"
    timeframe    : Tek TF modu için: "1m","5m","15m","30m","1h","4h","1d"
    multi_tf     : True → birden fazla TF çekilir (daha kapsamlı analiz).
    timeframes   : multi_tf=True iken kullanılacak TF listesi.
                   Varsayılan: ["1d", "4h", "1h"]
    reset        : True → konuşma geçmişini sıfırla.

    Örnekler
    --------
    live_chat("BTC long setup var mı?")
    live_chat("ETH analiz et", symbol="ETHUSDT", timeframe="1h")
    live_chat("SOL yapısı nasıl?", symbol="SOLUSDT", multi_tf=True)
    live_chat("BTC tam analiz", multi_tf=True, timeframes=["1d","4h","1h","15m"])
    """
    if multi_tf:
        tfs = timeframes or ["1d", "4h", "1h"]
        print(f"[{symbol}  {' + '.join(t.upper() for t in tfs)}  verisi alınıyor...]")
        ctx = fetch_multi_tf_context(symbol, timeframes=tfs)
    else:
        print(f"[{symbol} {timeframe.upper()} verisi alınıyor...]")
        ctx = fetch_context(symbol, timeframe=timeframe)

    print(ctx)
    print("─" * 60)
    return chat(user_message, market_context=ctx, reset=reset, **kwargs)


print("✅ live_chat() hazır")
print()
print("  Tek TF  : live_chat('BTC long uygun mu?')")
print("  Çoklu TF: live_chat('BTC yapısı?', multi_tf=True)")
print("  ETH     : live_chat('ETH analiz', symbol='ETHUSDT', timeframe='1h')")
print("  Full    : live_chat('Tam analiz', multi_tf=True, timeframes=['1d','4h','1h','15m'])")

In [ ]:
# --- Sohbet kutusu ---
# Mesajini asagiya yaz, hucreyi calistir. Tekrar yaz, tekrar calistir.
# Yeni konuya gecmek icin reset=True yap.

mesaj = "BTC 73k'yı kırarsa ne olur sence?"

response = chat(mesaj, reset=True)
print(response)

[sifirlandı]


---

## Utility — Update repo without reloading the model

In [ ]:
# Run this cell whenever you push prompt or code changes from VS Code.
# The model stays loaded — only the Python source is refreshed.
git_pull_latest()

# Reload the trading_prompt module to pick up any changes
import importlib, src.trading_prompt as _tp
importlib.reload(_tp)
from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages
print("Prompt module reloaded.")

Updating 0fa805e..d8d6dc4
Fast-forward
 setup.ipynb | 247 +++++++++++++++++++++++++++++++++++++++---------------------
 1 file changed, 162 insertions(+), 85 deletions(-)
From https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant
   0fa805e..d8d6dc4  main       -> origin/main
Repo ready at /content/unsloth-qwen35-trading-assistant
Prompt module reloaded.
